# Vehicle Detection, Tracking & Counting — Roboflow + YOLOv8

This is a **practical implementation notebook**, separate from the YOLO_family teaching notebook. Instead of building mechanisms from scratch, this uses production tools directly: **Roboflow** for dataset management, **Ultralytics YOLOv8** for detection, and a from-scratch **centroid tracker** for counting objects crossing a line — the same category of project as your existing YOLOv8 + ByteTrack tracking work, but built end-to-end here so you can see every piece.

**Pipeline:** Roboflow dataset → YOLOv8 (pretrained or fine-tuned) → per-frame detections → centroid tracking (links detections across frames into consistent IDs) → line-crossing counting.


## Step 1: Install dependencies

In [ ]:
# ultralytics: YOLOv8 training/inference
# roboflow: dataset download in YOLO-ready format
!pip install ultralytics roboflow -q


## Step 2: Get a dataset from Roboflow

**Why Roboflow here:** raw datasets are often inconsistent (mixed image sizes, no train/val/test split, wrong annotation format for your target model). Roboflow's export step fixes all of this in one click — you pick "YOLOv8" as the export format, and it hands you a ready-to-use folder with a `data.yaml` file already pointing at correctly-split, correctly-labeled data.

**To get your own API key:** sign up at roboflow.com (free tier is enough for this), open any public dataset (or upload your own images + draw boxes), click **Export**, choose **YOLOv8** format, and copy the generated code snippet — it will look like the cell below, just with your own `api_key` and project details filled in.

**No Roboflow account yet?** Skip to Step 2b below, which uses a small public vehicle dataset directly, no account needed, so you're not blocked while your account/project gets set up.


In [ ]:
# --- Option A: Your own Roboflow dataset ---
# Replace the placeholders below with YOUR values from the Roboflow export snippet.
# This cell is safe to skip (just don't run it) if you're using Option B instead.

from roboflow import Roboflow

ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"   # from roboflow.com account settings
WORKSPACE = "your-workspace-name"         # from your Roboflow project URL
PROJECT = "your-project-name"             # from your Roboflow project URL
VERSION = 1                               # dataset version number

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

print(f"Dataset downloaded to: {dataset.location}")
print("This folder contains train/valid/test splits and a data.yaml file YOLOv8 reads directly.")


### Step 2b: No Roboflow account? Use a pretrained model directly instead

If you don't have a custom dataset yet, you can skip training entirely and use YOLOv8's COCO-pretrained weights, which already recognize vehicles (car, truck, bus, motorcycle) out of the box. This lets you build and test the full tracking/counting pipeline today, and swap in a fine-tuned model later once your Roboflow dataset is ready.


In [ ]:
from ultralytics import YOLO

# yolov8n.pt = "nano" size -- smallest/fastest variant, good for quick testing.
# This downloads pretrained COCO weights (80 classes, including vehicles) automatically.
model = YOLO('yolov8n.pt')

# COCO class IDs for vehicles we care about: 2=car, 3=motorcycle, 5=bus, 7=truck
VEHICLE_CLASS_IDS = [2, 3, 5, 7]
print("Model loaded. Vehicle class names:", [model.names[i] for i in VEHICLE_CLASS_IDS])


## Step 3 (optional): Fine-tune on your Roboflow dataset

Only run this if you downloaded a custom dataset in Step 2 (Option A). This is a real training run — it needs a GPU (Colab: Runtime → Change runtime type → GPU) and will take real time depending on dataset size and epoch count.


In [ ]:
# Fine-tune YOLOv8 on your Roboflow-exported dataset.
# data: path to the data.yaml Roboflow generated (inside dataset.location from Step 2)
# epochs: how many full passes over the training data
# imgsz: input resolution YOLOv8 resizes images to internally

# results = model.train(
#     data=f"{dataset.location}/data.yaml",
#     epochs=50,
#     imgsz=640
# )

# After training, your fine-tuned weights are saved automatically at:
# runs/detect/train/weights/best.pt
# Load them for inference like this:
# model = YOLO('runs/detect/train/weights/best.pt')

print("Uncomment the lines above once your Roboflow dataset is ready.")


## Step 4: The Centroid Tracker (built from scratch)

YOLOv8 detects objects **per frame independently** — it has no concept of "this car in frame 10 is the same car as in frame 9." To count vehicles crossing a line, we need to **link detections across frames** into consistent tracked identities. This is a simplified version of what ByteTrack (which you've already used) does internally.

**The core idea:** for each new frame's detections, match them to existing tracked objects by finding the closest previous position (nearest centroid). If nothing is close enough, it's a new object; assign it a new ID.


In [ ]:
import numpy as np

class CentroidTracker:
    def __init__(self, max_distance=50):
        # max_distance: how far (in pixels) a detection can be from a previous
        # position and still be considered "the same object" between frames.
        self.next_id = 0
        self.objects = {}  # maps: tracked_id -> (x, y) centroid
        self.max_distance = max_distance

    def update(self, detections):
        # detections: list of (x, y) centroid tuples from the CURRENT frame
        updated_ids = {}
        for det in detections:
            best_id, best_dist = None, self.max_distance
            # Find the closest EXISTING tracked object to this new detection
            for oid, pos in self.objects.items():
                dist = np.linalg.norm(np.array(det) - np.array(pos))
                if dist < best_dist:
                    best_id, best_dist = oid, dist
            # No existing object close enough -> this is a genuinely new object
            if best_id is None:
                best_id = self.next_id
                self.next_id += 1
            updated_ids[best_id] = det
        self.objects = updated_ids
        return self.objects


def check_line_crossing(prev_y, curr_y, line_y):
    # Returns True exactly when an object's y-coordinate crosses from
    # ABOVE the line to AT-OR-BELOW the line between two consecutive frames.
    return prev_y < line_y <= curr_y


# --- Verify it works on a synthetic example before using real video ---
tracker = CentroidTracker()
line_y = 300
count = 0
prev_positions = {}

synthetic_frames = [
    [(100, 250)],
    [(102, 280)],
    [(105, 310)],  # crosses the line between this frame and the previous one
    [(108, 340)],
]

for frame_num, detections in enumerate(synthetic_frames):
    objects = tracker.update(detections)
    for oid, (x, y) in objects.items():
        if oid in prev_positions:
            prev_y = prev_positions[oid][1]
            if check_line_crossing(prev_y, y, line_y):
                count += 1
                print(f"Frame {frame_num}: object {oid} crossed the line! Running count: {count}")
        prev_positions[oid] = (x, y)

print(f"\nFinal count on synthetic test: {count} (expected: 1)")


## Step 5: Full Pipeline on Real Video

This combines everything: YOLOv8 detects vehicles in each frame → centroids get tracked across frames → crossings of a counting line get tallied per vehicle class.

Upload any traffic video to Colab (or mount Drive) and set `VIDEO_PATH` below.


In [ ]:
import cv2

VIDEO_PATH = "your_video.mp4"   # <-- set this to your uploaded video path
LINE_Y = 300                     # y-coordinate of the counting line (tune to your video's resolution)
CONFIDENCE_THRESHOLD = 0.4

tracker = CentroidTracker(max_distance=60)
prev_positions = {}
counts_by_class = {cid: 0 for cid in VEHICLE_CLASS_IDS}

cap = cv2.VideoCapture(VIDEO_PATH)
frame_num = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLOv8 inference on this single frame.
    # verbose=False keeps it from printing a log line per frame.
    results = model(frame, verbose=False)[0]

    # Collect (centroid, class_id) for vehicle-class detections only,
    # filtered by our confidence threshold.
    detections = []
    det_classes = {}
    for box in results.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        if cls_id in VEHICLE_CLASS_IDS and conf >= CONFIDENCE_THRESHOLD:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            centroid = ((x1 + x2) / 2, (y1 + y2) / 2)
            detections.append(centroid)
            det_classes[centroid] = cls_id

    # Update tracker with this frame's detections
    tracked_objects = tracker.update(detections)

    # Check each tracked object for a line crossing since last frame
    for oid, (x, y) in tracked_objects.items():
        if oid in prev_positions:
            prev_y = prev_positions[oid][1]
            if check_line_crossing(prev_y, y, LINE_Y):
                cls_id = det_classes.get((x, y))
                if cls_id is not None:
                    counts_by_class[cls_id] += 1
        prev_positions[oid] = (x, y)

    # Draw the counting line and detections for visual confirmation
    cv2.line(frame, (0, LINE_Y), (frame.shape[1], LINE_Y), (0, 255, 255), 2)
    for (x, y) in detections:
        cv2.circle(frame, (int(x), int(y)), 4, (0, 0, 255), -1)

    frame_num += 1

cap.release()

print("Final counts by vehicle class:")
for cid, cnt in counts_by_class.items():
    print(f"  {model.names[cid]}: {cnt}")


## Notes

- **Tune `LINE_Y`** to match where you actually want to count crossings in your specific video's resolution.
- **`max_distance` in the tracker** controls how much movement between frames still counts as "the same object" — too small and fast-moving vehicles get double-counted as new objects; too large and separate nearby vehicles get merged into one ID.
- This centroid tracker is intentionally simple (nearest-neighbor matching only) — it will struggle with vehicles that cross paths or get occluded. ByteTrack (which you've already used in your other project) solves this more robustly using motion prediction (Kalman filters) and a two-stage matching process — worth revisiting that project's tracker code side-by-side with this one to see exactly what extra robustness it buys you.
- Once your Roboflow dataset is ready and you've fine-tuned in Step 3, swap the model load in Step 2b for your `best.pt` weights — everything downstream (tracking, counting, visualization) works unchanged.
